# MD Feature Generation

## Purpose
Generate MD-derived feature columns from trajectory-side analysis outputs for downstream model-building tables.

## Input
- Primary input table: CSV with docking scores and RDKit descriptors (output of the descriptor calculation step)
- Trajectory/root directories referenced in the notebook `CFG`

## Output
- CSV table with MD-derived feature columns appended

## Run Order
1. Descriptor step (notebook 01)
2. This notebook
3. Pose labeling / feature selection / model notebook steps

In [ ]:
# ================================================================
# Cell 1: Libraries
# ================================================================

import os, re, glob, logging, warnings
import numpy as np
import pandas as pd
import MDAnalysis as mda
from MDAnalysis.lib import distances as mda_dist
from MDAnalysis.analysis import rms as mda_rms
from MDAnalysis.analysis import align
from rdkit import RDLogger

# ---- Warning and logging control ----
warnings.filterwarnings("ignore", message=".*pkg_resources is deprecated.*")
warnings.filterwarnings("ignore", message=".*MDAnalysis\\.topology\\.tables has been moved.*")
warnings.filterwarnings("ignore", message=".*standardization could not be completed.*")
warnings.filterwarnings("ignore", message=".*Reload offsets from trajectory.*")
logging.getLogger("MDAnalysis").setLevel(logging.ERROR)
RDLogger.DisableLog("rdApp.*")

# ---- ProLIF (required) ----
try:
    from prolif import Fingerprint
except Exception as e:
    raise ImportError(
        "[FATAL] ProLIF could not be imported. Please install it with `pip install prolif`."
    ) from e


In [ ]:
# ================================================================
# Cell 2: Configuration and feature-builder helpers
# ================================================================

# ==========================================================
# Configuration — update the paths in CFG before running
#   csv_in:             input CSV from the descriptor step
#   csv_out:            destination CSV for this notebook
#   traj_root:          root directory containing per-PDB MD trajectory sub-directories
#   processed_pdb_root: directory with processed PDB files (used for residue-offset detection)
# ==========================================================
CFG = {
    # Input/output paths
    "csv_in":            "/path/to/input.csv",
    "csv_out":           "/path/to/output.csv",
    "traj_root":         "/path/to/analysis/trajectories",
    "processed_pdb_root":"/path/to/analysis/proteins/",

    # Fixed filenames (following making-it-rain protocol output)
    "topo_file": "SYS_gaff2_nw.prmtop",
    "traj_file": "prot_lig_prod1-5_nw.xtc",

    # Geometric parameters and sampling
    "contact_cutoff": 4.5,        # Å (atom-pair contact criterion)
    "stride_contacts": 10,        # frame stride for contact computation (speed-up)
    "stride_prolif":  10,         # frame stride for ProLIF analysis
    "prolif_shell_radius": 14.0,  # Residue extraction range around ligand

    "limit_num_pdbs": None,  # set to an integer to limit the number of PDBs processed (for testing)

    # Binding site definition (PDB standard residue numbers)
    "site_core": {
        "site1": sorted({150,195,199,214,218,222,242,257,238,260,264}),
        "site2": sorted({410,411,414,430,433,452,453,488,489}),
        "site3": sorted({142,137,161,138,186,117})
    },
}

# Column index for value column per CSV file
PER_FILE_VALUE_COL = {
    "rmsd_ca.csv": 1,
    "radius_gyration.csv": 1,
    "distance.csv": 1,
    "Interaction_energy_eelec.csv": 1,
    "Interaction_energy_evdw.csv": 1,
    "rmsf_ca.csv": 2,
}

_OFFSET_CACHE = {}

# ==========================================================
# Helpers
# ==========================================================
def norm_ligand_id(s):
    """
    Normalize ligand_id to 'poseXX' format.
    Strings not containing 'pose' are returned as-is and filtered out later.
    """
    if not isinstance(s, str):
        return None
    s = s.strip()
    m = re.match(r"(?:gnina[_\- ]?)?pose[_\- ]?(\d+)", s, flags=re.IGNORECASE)
    if m:
        return f"pose{int(m.group(1)):02d}"
    return s

def read_csv_safe(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return None

def pick_value_col_by_rule(df: pd.DataFrame, filename: str):
    base = os.path.basename(filename)
    if base in PER_FILE_VALUE_COL:
        col_idx = PER_FILE_VALUE_COL[base]
        if df.shape[1] > col_idx:
            return df.columns[col_idx]
    return df.columns[1] if df.shape[1] >= 2 else None

def summarize_series(y, t=None, prefix=""):
    """
    Compute descriptive statistics from a time series (general-purpose).
    11 summary statistics: mean, std, median, min, max, p10, p90, last, slope, delta, drift
    """
    y = np.asarray(y, dtype=float)
    keys = ["mean","std","median","min","max","p10","p90","last","slope","delta","drift"]
    if y.size == 0 or np.all(np.isnan(y)):
        return {f"{prefix}{k}": np.nan for k in keys}

    out = {
        f"{prefix}mean":   float(np.nanmean(y)),
        f"{prefix}std":    float(np.nanstd(y)),
        f"{prefix}median": float(np.nanmedian(y)),
        f"{prefix}min":    float(np.nanmin(y)),
        f"{prefix}max":    float(np.nanmax(y)),
        f"{prefix}p10":    float(np.nanpercentile(y,10)),
        f"{prefix}p90":    float(np.nanpercentile(y,90)),
        f"{prefix}last":   float(y[~np.isnan(y)][-1]) if np.any(~np.isnan(y)) else np.nan,
    }

    x = np.arange(len(y), dtype=float) if t is None else np.asarray(t, dtype=float)
    mask = (~np.isnan(y)) & (~np.isnan(x))
    if mask.sum() >= 2:
        out[f"{prefix}slope"] = float(np.polyfit(x[mask], y[mask], 1)[0])
    else:
        out[f"{prefix}slope"] = np.nan

    if y.size >= 2:
        mid = len(y) // 2
        y_early = y[:mid]
        y_late = y[mid:]
        mean_early = float(np.nanmean(y_early)) if y_early.size > 0 else np.nan
        mean_late = float(np.nanmean(y_late)) if y_late.size > 0 else np.nan
        out[f"{prefix}delta"] = mean_late - mean_early if not (np.isnan(mean_early) or np.isnan(mean_late)) else np.nan
        out[f"{prefix}drift"] = out[f"{prefix}last"] - mean_early if not (np.isnan(out[f"{prefix}last"]) or np.isnan(mean_early)) else np.nan
    else:
        out[f"{prefix}delta"] = np.nan
        out[f"{prefix}drift"] = np.nan

    return out

def summarize_series_selected(y, prefix="", keys=None):
    """
    Extract only the requested statistics from summarize_series() output.
    """
    full = summarize_series(y, prefix=prefix)
    if keys is None:
        return full
    return {f"{prefix}{k}": full.get(f"{prefix}{k}", np.nan) for k in keys}

def parse_timeseries_csv(path, prefix):
    """
    Load an external CSV time-series file and return summary statistics (all 11).
    """
    df = read_csv_safe(path)
    keys = ["mean","std","median","min","max","p10","p90","last","slope","delta","drift"]
    if df is None or df.empty:
        return {f"{prefix}{k}": np.nan for k in keys}
    vcol = pick_value_col_by_rule(df, path)
    if vcol is None:
        return {f"{prefix}{k}": np.nan for k in keys}
    y = pd.to_numeric(df[vcol], errors="coerce")
    return summarize_series(y, prefix=prefix)

def parse_mmpbsa_final(path):
    """Extract free energy values from a MMPBSA.py output file."""
    if not os.path.isfile(path):
        return {}
    txt = open(path, "r", errors="ignore").read()
    out = {}

    gb = re.search(r"GENERALIZED BORN:(.*?)(?:POISSON BOLTZMANN:|$)", txt, flags=re.S)
    if gb:
        m = re.search(r"DELTA TOTAL\s+([-\d\.Ee+]+)\s+([-\d\.Ee+]+)", gb.group(1))
        if m:
            out["md_mmgbsa_delta_total"] = float(m.group(1))
            out["md_mmgbsa_std"] = float(m.group(2))

    pb = re.search(r"POISSON BOLTZMANN:(.*?)(?:={5,}|$)", txt, flags=re.S)
    if pb:
        m = re.search(r"DELTA TOTAL\s+([-\d\.Ee+]+)\s+([-\d\.Ee+]+)", pb.group(1))
        if m:
            out["md_mmpbsa_delta_total"] = float(m.group(1))
            out["md_mmpbsa_std"] = float(m.group(2))

    return out

def autodetect_ligand_atoms(u: mda.Universe):
    """Auto-detect ligand atom group within a Universe."""
    nonprot = u.select_atoms(
        "not protein and not resname HOH WAT and "
        "not resname NA K CL MG CA and not resname Na+ K+ Cl- Mg2+ Ca2+"
    )
    if nonprot.n_atoms == 0:
        return None
    best_res, best_heavy = None, -1
    for res in nonprot.residues:
        heavy = res.atoms.select_atoms("prop mass > 1.5").n_atoms
        if heavy > best_heavy:
            best_heavy, best_res = heavy, res
    return best_res.atoms if best_res is not None else None

def pick_top_files(sd_path: str):
    """Check for topology and trajectory files and return their paths."""
    top = os.path.join(sd_path, CFG["topo_file"])
    trj = os.path.join(sd_path, CFG["traj_file"])
    if not (os.path.isfile(top) and os.path.isfile(trj)):
        return None, None
    return top, trj

def get_resid_offset(pdb_id: str) -> int:
    """Auto-calculate residue index offset from a processed PDB file."""
    key = str(pdb_id)
    if key in _OFFSET_CACHE:
        return _OFFSET_CACHE[key]
    p = os.path.join(CFG["processed_pdb_root"], f"{key}-processed.pdb")
    if not os.path.isfile(p):
        raise FileNotFoundError(f"[OFFSET][FATAL] processed PDB not found: {p}")
    resseq = None
    with open(p, "r") as f:
        for line in f:
            if line.startswith("ATOM"):
                resseq = int(line[22:26].strip())
                break
    if resseq is None:
        raise RuntimeError(f"No ATOM found in {p}")
    off = int(resseq - 1)
    _OFFSET_CACHE[key] = off
    print(f"[OFFSET] {pdb_id}: offset={off}")
    return off

def get_site_resids_for(pdb_id: str, site_name: str):
    """Convert PDB-standard residue numbers to Universe (post-offset) numbering."""
    base = CFG["site_core"].get(site_name, [])
    if not base:
        return []
    off = get_resid_offset(pdb_id)
    return [int(r - off) for r in base]

def _entropy_from_counts(counts: dict) -> float:
    arr = np.array([max(0.0, float(v)) for v in counts.values()], dtype=float)
    tot = arr.sum()
    if tot <= 0:
        return 0.0
    p = arr / tot
    return float(-np.sum(p * np.log(p + 1e-12)))

def _runs_from_bool_series(b: np.ndarray):
    """
    Return a list of consecutive True run lengths from a boolean array.
    Example: [True, True, False, True] -> [2, 1]
    """
    if b.size == 0:
        return []
    runs, cnt = [], 0
    for v in b.astype(bool):
        if v:
            cnt += 1
        elif cnt > 0:
            runs.append(cnt)
            cnt = 0
    if cnt > 0:
        runs.append(cnt)
    return runs

# ==========================================================
# Core Calculation Functions
# ==========================================================
def compute_ligand_rmsd(sd_path, stride):
    """
    Compute ligand RMSD relative to the initial structure.
    Aligns on protein Ca atoms first, then computes ligand RMSD.
    Returns 11 summary statistics (see summarize_series).
    """
    top, trj = pick_top_files(sd_path)
    if not top:
        return summarize_series(np.array([]), prefix="md_rmsd_lig_")

    try:
        u = mda.Universe(top, trj)
        lig = autodetect_ligand_atoms(u)
        if lig is None:
            return summarize_series(np.array([]), prefix="md_rmsd_lig_")

        u_ref = mda.Universe(top, trj)
        u_ref.trajectory[0]
        lig_ref = autodetect_ligand_atoms(u_ref)
        if lig_ref is None:
            return summarize_series(np.array([]), prefix="md_rmsd_lig_")
        lig_ref_pos = lig_ref.positions.copy()

        rmsds = []
        for _ in u.trajectory[::max(1, stride)]:
            align.alignto(u, u_ref, select="protein and name CA")
            rmsd_val = mda_rms.rmsd(lig.positions, lig_ref_pos, center=False, superposition=False)
            rmsds.append(rmsd_val)

        y = np.asarray(rmsds, dtype=float)
        return summarize_series(y, prefix="md_rmsd_lig_")

    except Exception as e:
        print(f"[LIGAND_RMSD ERROR] {sd_path}: {e}")
        import traceback
        traceback.print_exc()
        return summarize_series(np.array([]), prefix="md_rmsd_lig_")

def compute_contacts(sd_path, cutoff, stride):
    """
    Compute time series of protein-ligand atom-pair contact counts.

    Additional:
    - md_contact_unique_residues: number of distinct residues contacted during MD (diversity)
    - md_contact_residue_entropy: entropy of the contacted-residue distribution

    Existing:
    - md_contact_*: 11 summary statistics of contact atom-pair counts
    - md_contact_per_atom_*: 11 summary statistics normalized by number of ligand atoms
    """
    top, trj = pick_top_files(sd_path)
    if not top:
        return {}
    try:
        u = mda.Universe(top, trj)
        prot = u.select_atoms("protein")
        lig = autodetect_ligand_atoms(u)
        if lig is None or prot.n_atoms == 0:
            return {}

        prot_resids = prot.resids
        counts = []
        unique_resid_set = set()
        resid_frame_counts = {}

        for _ in u.trajectory[::max(1, stride)]:
            d = mda_dist.distance_array(prot.positions, lig.positions)
            counts.append((d < cutoff).sum())

            min_d = d.min(axis=1)
            contact_atoms = min_d < cutoff
            if np.any(contact_atoms):
                resids = np.unique(prot_resids[contact_atoms]).tolist()
                for r in resids:
                    unique_resid_set.add(int(r))
                    resid_frame_counts[int(r)] = resid_frame_counts.get(int(r), 0) + 1

        y = np.asarray(counts, dtype=float)

        out = summarize_series(y, prefix="md_contact_")
        out.update(summarize_series(y / max(1, lig.n_atoms), prefix="md_contact_per_atom_"))

        out["md_contact_unique_residues"] = float(len(unique_resid_set))

        if len(resid_frame_counts) == 0:
            out["md_contact_residue_entropy"] = 0.0
        else:
            c = np.asarray(list(resid_frame_counts.values()), dtype=float)
            p = c / c.sum()
            out["md_contact_residue_entropy"] = float(-np.sum(p * np.log(p + 1e-12)))

        return out
    except Exception:
        return {}

def compute_rmsf_protein(sd_path: str, pdb_id: str):
    """
    Aggregate whole-protein RMSF from rmsf_ca.csv (no site-specific calculations).
    Statistics: mean, max, std, p90
    """
    csv_path = os.path.join(sd_path, "rmsf_ca.csv")
    empty = {
        "md_rmsf_protein_mean": np.nan,
        "md_rmsf_protein_max": np.nan,
        "md_rmsf_protein_std": np.nan,
        "md_rmsf_protein_p90": np.nan,
    }
    if not os.path.isfile(csv_path):
        return empty
    try:
        df = pd.read_csv(csv_path)
        if df.shape[1] >= 3:
            rmsf_col_idx = 2
        elif df.shape[1] >= 2:
            rmsf_col_idx = 1
        else:
            return empty

        rmsf_vals = pd.to_numeric(df.iloc[:, rmsf_col_idx], errors="coerce").to_numpy()
        rmsf_vals = rmsf_vals[~np.isnan(rmsf_vals)]
        if rmsf_vals.size == 0:
            return empty

        return {
            "md_rmsf_protein_mean": float(np.nanmean(rmsf_vals)),
            "md_rmsf_protein_max": float(np.nanmax(rmsf_vals)),
            "md_rmsf_protein_std": float(np.nanstd(rmsf_vals)),
            "md_rmsf_protein_p90": float(np.nanpercentile(rmsf_vals, 90)),
        }
    except Exception:
        return empty

def compute_prolif_features(sd_path, stride):
    """
    Compute ProLIF-based interaction features.
    - Success (df obtained): fill missing interaction types with 0 (no missing columns)
    - Failure (exception / empty df): all ProLIF-derived columns are set to NaN
      (but columns are always created)

    Note: in this version, statistics for count_* are limited to
          "mean/median/max/p90/last". std/slope/delta/drift are not generated.
    """
    freq_types_all = ["hydrophobic", "hbond", "aromatic", "ionic", "vdw", "metal", "halogen", "aromaticgeom"]
    count_types_main = ["hydrophobic", "hbond", "aromatic", "ionic", "vdw"]

    # Statistics to generate (user-specified)
    stat_keys = ["mean", "median", "max", "p90", "last"]

    def _init_success_zero():
        res = {
            "md_prolif_any_contact_freq": 0.0,
            "md_prolif_contact_run_mean": 0.0,
            "md_prolif_contact_run_max": 0.0,
            "md_prolif_interaction_entropy": 0.0,
            "md_prolif_polar_ratio": 0.0,
            "md_prolif_aromatic_ratio": 0.0,
        }
        for k in stat_keys:
            res[f"md_prolif_any_contact_count_{k}"] = 0.0
        for t in freq_types_all:
            res[f"md_prolif_{t}_freq"] = 0.0
        for t in count_types_main:
            for k in stat_keys:
                res[f"md_prolif_{t}_count_{k}"] = 0.0
        return res

    def _init_fail_nan():
        res = {
            "md_prolif_any_contact_freq": np.nan,
            "md_prolif_contact_run_mean": np.nan,
            "md_prolif_contact_run_max": np.nan,
            "md_prolif_interaction_entropy": np.nan,
            "md_prolif_polar_ratio": np.nan,
            "md_prolif_aromatic_ratio": np.nan,
        }
        for k in stat_keys:
            res[f"md_prolif_any_contact_count_{k}"] = np.nan
        for t in freq_types_all:
            res[f"md_prolif_{t}_freq"] = np.nan
        for t in count_types_main:
            for k in stat_keys:
                res[f"md_prolif_{t}_count_{k}"] = np.nan
        return res

    top, trj = pick_top_files(sd_path)
    if not top:
        return _init_fail_nan()

    try:
        u = mda.Universe(top, trj)
        lig = autodetect_ligand_atoms(u)
        if lig is None:
            return _init_fail_nan()

        R = float(CFG["prolif_shell_radius"])
        prot_subset = u.select_atoms(f"protein and byres around {R} group ligand", ligand=lig)

        if prot_subset.n_atoms == 0:
            return _init_success_zero()

        fp = Fingerprint()
        fp.run(u.trajectory[::max(1, stride)], lig, prot_subset)
        df = fp.to_dataframe()
        if df is None or df.empty:
            return _init_fail_nan()

        # Success path: start from zero and fill in computed values
        res = _init_success_zero()
        df_bool = df.astype(bool)

        # any_contact_freq
        any_contact = df_bool.any(axis=1)
        res["md_prolif_any_contact_freq"] = float(any_contact.mean())

        # any_contact_count_* (limited statistics)
        any_series = df_bool.sum(axis=1).astype(int)
        res.update(summarize_series_selected(any_series, prefix="md_prolif_any_contact_count_", keys=stat_keys))

        # run-length
        runs = _runs_from_bool_series(any_contact.to_numpy())
        res["md_prolif_contact_run_mean"] = float(np.mean(runs)) if runs else 0.0
        res["md_prolif_contact_run_max"]  = float(np.max(runs)) if runs else 0.0

        # If not MultiIndex, return with per-type stats as zero
        if not (isinstance(df.columns, pd.MultiIndex) and "interaction" in df.columns.names):
            return res

        # interaction -> category mapping
        type_name_mapping = {
            "Hydrophobic": "hydrophobic",
            "HBAcceptor": "hbond",
            "HBDonor": "hbond",
            "PiStacking": "aromatic",
            "CationPi": "aromatic",
            "VdWContact": "vdw",
            "Cationic": "ionic",
            "Anionic": "ionic",
            "PiCation": "ionic",
            "MetalAcceptor": "metal",
            "MetalDonor": "metal",
            "XBAcceptor": "halogen",
            "XBDonor": "halogen",
            "FaceToFace": "aromaticgeom",
            "EdgeToFace": "aromaticgeom",
        }

        interaction_types = df.columns.get_level_values("interaction").unique()
        n_frames = df_bool.shape[0]

        presence = {t: 0.0 for t in freq_types_all}
        counts = {t: np.zeros(n_frames, dtype=int) for t in count_types_main}

        for itype in interaction_types:
            mask = (df.columns.get_level_values("interaction") == itype)
            sub_bool = df_bool.loc[:, mask]
            cat = type_name_mapping.get(itype, str(itype).lower())

            pres = float(sub_bool.any(axis=1).mean()) if sub_bool.shape[1] > 0 else 0.0
            if cat in presence:
                presence[cat] = max(presence[cat], pres)

            if cat in counts:
                counts[cat] += sub_bool.sum(axis=1).astype(int).to_numpy()

        # freq
        for t in freq_types_all:
            res[f"md_prolif_{t}_freq"] = float(presence[t])

        # interaction entropy
        pres_nonzero = {k: v for k, v in presence.items() if v > 0}
        if len(pres_nonzero) > 1:
            res["md_prolif_interaction_entropy"] = _entropy_from_counts(pres_nonzero)
        else:
            res["md_prolif_interaction_entropy"] = 0.0

        # ratios (based on mean counts)
        mean_counts = {k: float(np.mean(v)) for k, v in counts.items()}
        total_mean = float(sum(mean_counts.values()))
        if total_mean > 0:
            polar_mean = mean_counts.get("hbond", 0.0) + mean_counts.get("ionic", 0.0)
            aromatic_mean = mean_counts.get("aromatic", 0.0)
            res["md_prolif_polar_ratio"] = float(polar_mean / total_mean)
            res["md_prolif_aromatic_ratio"] = float(aromatic_mean / total_mean)
        else:
            res["md_prolif_polar_ratio"] = 0.0
            res["md_prolif_aromatic_ratio"] = 0.0

        # count stats (limited statistics only)
        for t in count_types_main:
            res.update(summarize_series_selected(counts[t], prefix=f"md_prolif_{t}_count_", keys=stat_keys))

        return res

    except Exception:
        return _init_fail_nan()

# ==========================================================
# Main Collector
# ==========================================================
def collect_md_features(traj_root, target_pdbs=None):
    """
    Scan all trajectory directories and collect features for pose folders only.
    """
    feats = {}
    pdb_dirs = [d for d in glob.glob(os.path.join(traj_root, "*")) if os.path.isdir(d)]

    for pdb_path in sorted(pdb_dirs):
        pdb_id = os.path.basename(pdb_path)
        if target_pdbs is not None and pdb_id not in set(target_pdbs):
            continue

        all_subdirs = [d for d in glob.glob(os.path.join(pdb_path, "*")) if os.path.isdir(d)]
        subdirs = [d for d in all_subdirs if "pose" in os.path.basename(d).lower()]
        if not subdirs:
            continue

        print(f"[PDB] Processing {pdb_id} ({len(subdirs)} pose directories found)")

        for sd_path in sorted(subdirs):
            sub = os.path.basename(sd_path)
            f = {}

            # generic CSV summaries (all 11 statistics)
            csv_map = {
                "rmsd_ca.csv": "md_rmsd_ca_",
                "radius_gyration.csv": "md_rg_",
                "distance.csv": "md_distance_",
                "Interaction_energy_eelec.csv": "md_eelec_",
                "Interaction_energy_evdw.csv": "md_evdw_",
            }
            for fname, prefix in csv_map.items():
                f.update(parse_timeseries_csv(os.path.join(sd_path, fname), prefix=prefix))

            # ligand RMSD (all 11 statistics)
            f.update(compute_ligand_rmsd(sd_path, stride=CFG["stride_contacts"]))

            # free energy, RMSF, contacts, ProLIF
            f.update(parse_mmpbsa_final(os.path.join(sd_path, "FINAL_RESULTS_MMPBSA.dat")))
            f.update(compute_rmsf_protein(sd_path, pdb_id=pdb_id))
            f.update(compute_contacts(sd_path, cutoff=CFG["contact_cutoff"], stride=CFG["stride_contacts"]))
            f.update(compute_prolif_features(sd_path, stride=CFG["stride_prolif"]))

            # md_distance_below series are not generated (removed per analysis design)
            # f.update(compute_distance_threshold_features(...))

            if f:
                feats[(pdb_id, sub)] = f

    return feats


In [ ]:
# ================================================================
# Cell 3: Main merge routine
# ================================================================

def run_md_feature_merge():
    """Merge MD-derived features into the descriptor table and save the result."""
    df_in = pd.read_csv(CFG["csv_in"])
    df_in["ligand_id_norm"] = df_in["ligand_id"].apply(norm_ligand_id)

    # Keep only docking-pose rows in the public workflow.
    initial_shape = df_in.shape
    df_in = df_in[df_in["ligand_id_norm"].str.startswith("pose", na=False)]
    removed_count = initial_shape[0] - df_in.shape[0]
    if removed_count > 0:
        print(f"[FILTER] Removed {removed_count} non-pose rows (kept {df_in.shape[0]} pose rows)")

    target_pdbs = None
    if CFG.get("limit_num_pdbs"):
        target_pdbs = df_in["pdb_id"].unique()[:CFG["limit_num_pdbs"]]

    md_feat = collect_md_features(CFG["traj_root"], target_pdbs=target_pdbs)
    rows = [{"pdb_id": pid, "ligand_id_folder": sub, **feat}
            for (pid, sub), feat in md_feat.items()]
    md_df = pd.DataFrame(rows)
    md_df["ligand_id_norm"] = md_df["ligand_id_folder"].apply(norm_ligand_id)

    merged = df_in.merge(
        md_df.drop(columns=["ligand_id_folder"], errors="ignore"),
        on=["pdb_id", "ligand_id_norm"],
        how="left",
    )
    merged = merged.drop(columns=["ligand_id_norm"], errors="ignore")

    # Remove RMSD minima that are trivially zero by construction.
    cols_to_drop = [c for c in ["md_rmsd_lig_min", "md_rmsd_ca_min"] if c in merged.columns]
    if cols_to_drop:
        merged = merged.drop(columns=cols_to_drop)
        print(f"[DROP] Removed columns: {cols_to_drop}")

    os.makedirs(os.path.dirname(CFG["csv_out"]) or ".", exist_ok=True)
    merged.to_csv(CFG["csv_out"], index=False)
    print(f"[OK] Wrote {CFG['csv_out']} (shape: {merged.shape})")
    return merged


In [ ]:
# ================================================================
# Cell 4: Execution
# ================================================================

run_md_feature_merge()
